# CWSB Stem Detector — one-run Colab pipeline

Trains a YOLO **detector** on the new `data/` dataset (640×640 stem close-ups,
Roboflow export, 423 train / 41 valid / 22 test, 3 box classes).

This is **not** a retrain of `contender_B.pt` (the whole-plant classifier).
It is the new stem-level model: it finds WSB evidence (bore holes / ridges) inside a close-up.

**Before running — two things:**
1. Runtime → Change runtime type → **T4 GPU** → Save. (If cell 1 prints no GPU, the session is on CPU. Changing the runtime wipes session files, so do it first.)
2. Set the real class names in **cell 3**. The labels use ids 0/1/2 but the export shipped no `data.yaml`, so the names below are guesses from box shapes.

**How to run:** upload `data.zip` (made by zipping the repo's `data/` folder) when cell 2 asks, then Runtime → Run all. Takes roughly 30–45 min on a T4. The last cell downloads `cwsb_detector_run.zip` with the weights and results.

In [ ]:
# Cell 1 - GPU check + install
!nvidia-smi -L || echo 'NO GPU - fix the runtime type before going on (this wipes files)'
%pip -q install ultralytics
import torch, random, numpy as np
SEED = 42  # project-wide convention
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# Cell 2 - get the data
# Option A (default): upload data.zip from your machine.
# Option B: put data.zip in Google Drive and uncomment the two Drive lines instead.
from pathlib import Path

# from google.colab import drive
# drive.mount('/content/drive'); !cp '/content/drive/MyDrive/data.zip' /content/

if not Path('/content/data.zip').exists():
    from google.colab import files
    up = files.upload()  # pick data.zip

!unzip -oq /content/data.zip -d /content/dataset   # -oq: never trust the interactive prompt

# tolerate either zip layout: dataset/train/... or dataset/data/train/...
DATA = Path('/content/dataset')
if not (DATA / 'train').exists():
    DATA = next(p for p in DATA.iterdir() if (p / 'train').exists())
!find {DATA} -name 'labels.cache' -delete   # stale caches from local runs
print('dataset root:', DATA)

In [ ]:
# Cell 3 - data.yaml  >>> SET THE REAL CLASS NAMES HERE <<<
# Ids come from the label .txt files. Current names are guesses:
#   0 -> small squarish boxes  (bore hole?)
#   1 -> larger elongated boxes (ridge / frass line?)
#   2 -> rare, only 19 boxes in the whole set
CLASS_NAMES = ['hole', 'ridge', 'class_2']

yaml_text = f"""path: {DATA}
train: train/images
val: valid/images
test: test/images
nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
Path('/content/data.yaml').write_text(yaml_text)
print(yaml_text)

In [ ]:
# Cell 4 - sanity check before burning GPU time
from collections import Counter
for split in ['train', 'valid', 'test']:
    imgs = list((DATA / split / 'images').glob('*'))
    lbls = list((DATA / split / 'labels').glob('*.txt'))
    boxes = Counter()
    for t in lbls:
        for line in t.read_text().splitlines():
            if line.strip():
                boxes[int(line.split()[0])] += 1
    print(f'{split:5s}  {len(imgs):3d} images  {len(lbls):3d} label files  boxes per class: {dict(sorted(boxes.items()))}')
# expected: train 423/422 (one background image), valid 41/41, test 22/22, classes 0/1/2 everywhere

In [ ]:
# Cell 5 - train
from ultralytics import YOLO
model = YOLO('yolo11s.pt')  # small: right size for ~400 training images
results = model.train(
    data='/content/data.yaml',
    epochs=150, patience=30,          # early stopping does the real epoch choice
    imgsz=640, batch=16, seed=SEED,
    project='/content/runs', name='cwsb_detector_v1', exist_ok=True,
)

In [ ]:
# Cell 6 - evaluate the best checkpoint on the held-out TEST split
best = YOLO('/content/runs/cwsb_detector_v1/weights/best.pt')
m = best.val(data='/content/data.yaml', split='test')
print('\nTEST  mAP50 %.3f   mAP50-95 %.3f' % (m.box.map50, m.box.map))
for i, name in enumerate(CLASS_NAMES):
    if i < len(m.box.maps):
        print(f'  {name:10s} mAP50-95 {m.box.maps[i]:.3f}')

In [ ]:
# Cell 7 - annotated predictions on the 22 test images (visual evidence)
best.predict(source=str(DATA / 'test' / 'images'), save=True, conf=0.25,
             project='/content/runs', name='test_predictions', exist_ok=True)
import glob
from IPython.display import Image as IPImage, display
for f in sorted(glob.glob('/content/runs/test_predictions/*.jpg'))[:4]:
    display(IPImage(f, width=480))  # first few inline; all of them go in the download zip

In [ ]:
# Cell 8 - package and download
# IMPORTANT (known pitfall): unzip this FLAT on the local machine -> models/v2_detector/
# with best.pt directly inside. Do not let it nest models/v2/models/v2/.
!mkdir -p /content/out/weights /content/out/training /content/out/test_predictions
!cp /content/runs/cwsb_detector_v1/weights/best.pt /content/out/weights/cwsb_detector_v1_best.pt
!cp /content/runs/cwsb_detector_v1/weights/last.pt /content/out/weights/cwsb_detector_v1_last.pt
!cp /content/runs/cwsb_detector_v1/results.csv /content/runs/cwsb_detector_v1/*.png /content/out/training/ 2>/dev/null || true
!cp /content/runs/test_predictions/*.jpg /content/out/test_predictions/
!cp /content/data.yaml /content/out/
!cd /content/out && zip -rq /content/cwsb_detector_run.zip .
from google.colab import files
files.download('/content/cwsb_detector_run.zip')

## After downloading

1. Unzip into the repo as `models/v2_detector/` — **flat**, `cwsb_detector_v1_best.pt` directly inside.
2. Log the run (mAP numbers, epochs, class names used) in `charan_working.md` and update `PROJECT_STATE.md`.
3. Caveats to carry into the write-up:
   - 22 test images is a tiny test set — treat mAP as indicative, same ±margin honesty as the classifier numbers.
   - Class 2 has ~19 boxes total; its per-class metric is noise. Consider merging it or collecting more.
   - This detector answers *"is there WSB evidence in this stem close-up"*. The whole-plant classifier and the framing gate are still separate pieces of the final app.